<a href="https://colab.research.google.com/github/vinodarya/LLMInference/blob/main/vllm_colab_qwen_vl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Running VLLM in Colab**

This notebook demonstrates how to run VLLM in colab, serving qwen-vl model.

Why this is significant:

1. Able to run multimodal large language model hosted in VLLM
2. Ollama is best for fast local development and single-user workflows, while VLLM is built for high-throughput, concurrent production deployments
3. Colab provides you free GPU and you can run without any local/cloud setup plus not spending any money

**Credit:** Abhishek Selokar repo https://github.com/selokarabhishek/ai-ml-tutorials/blob/main/qwen_vl_vllm_colab_serving_.ipynb

##**Step1:**
a. Setup environment to run code in GPU
* Goto "Runtime" menu
* Click on "Change runtime type"
* Select "T4 GPU"
* Click on "Save"

b. Validate GPU is enabled by running "!nvidia-smi"


In [1]:
!nvidia-smi

Sat Nov 21 05:31:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

##**Step 2:**

Install the required dependencies

In [2]:
!pip install vllm transformers qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 MB 32.3 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 132.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 MB 66.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.0/556.0 kB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 148.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 103.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 67.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 MB 66.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 68.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 44.0 MB/s  0:00:00
   ━━━━━━━

##**Step 3:**

Verify the qwen-vl architecture is recognized, this is important as the model will subsequently loaded and inferenced

In [1]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", trust_remote_code=True)
print(config.architectures)
# Expected: ['Qwen2_5_VLForConditionalGeneration']

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

['Qwen2_5_VLForConditionalGeneration']


##**Step 4**

Download and load the model

Note: It will ask to enable Huggingface access to the notebook

In [2]:
import os
os.environ["USE_TF"] = "0"          # <-- add this: stops transformers from ever importing TensorFlow
os.environ["VLLM_TORCH_COMPILE_LEVEL"] = "0"

from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    trust_remote_code=True,
    enforce_eager=True,
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    limit_mm_per_prompt={"image": 1},
        mm_processor_kwargs={"max_pixels": 802816},
)
print("Model loaded successfully.")

W1121 05:36:52.509000 2775 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1121 05:36:52.511000 2775 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1121 05:36:52.517000 2775 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=2775) INFO 11-21 05:37:17 [startup.py:96] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 1}, 'mm_processor_kwargs': {'max_pixels': 802816}, 'model': 'Qwen/Qwen2.5-VL-3B-Instruct'}
(MainProcess pid=2775) WARNING 11-21 05:37:17 [interface.py:1500] Unknown vLLM environment variable detected: VLLM_TORCH_COMPILE_LEVEL
(MainProcess pid=2775) INFO 11-21 05:37:36 [model.py:761] Resolved architecture: Qwen2_5_VLForConditionalGeneration
(MainProcess pid=2775) WARNING 11-21 05:37:36 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=2775) INFO 11-21 05:37:36 [model.py:2214] Using max model len 4096


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

(MainProcess pid=2775) INFO 11-21 05:37:39 [scheduler.py:354] Chunked prefill is enabled with max_num_batched_tokens=8192.


... (4 outputs left out)

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=3478) WARNING 11-21 05:41:13 [utils.py:345] Attempted to load weight embed_tokens.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=3478) WARNING 11-21 05:41:16 [utils.py:345] Attempted to load weight layers.0.input_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=3478) WARNING 11-21 05:41:16 [utils.py:345] Attempted to load weight layers.0.mlp.down_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=3478) WARNING 11-21 05:41:17 [utils.py:345] Attempted to load weight layers.0.post_attention_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=3478) WARNING 11-21 05:41:17 [utils.py:345] Attempted to load weight layers.0.self_attn.o_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=3478) WARNING 11-21 05:41:17 [utils.py:345] Attempted to load weight laye

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/video_processing_qwen2_vl.py:156: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


(EngineCore pid=3478) INFO 11-21 05:41:49 [encoder_runner.py:131] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 2 video items of the maximum feature size.
(EngineCore pid=3478) INFO 11-21 05:42:11 [gpu_worker.py:795] Available KV cache memory: 3.97 GiB
(EngineCore pid=3478) INFO 11-21 05:42:11 [qwen_vl_triton_warmup.py:102] Warmed M-RoPE Triton kernels.
(EngineCore pid=3478) INFO 11-21 05:42:46 [gpu_worker.py:1175] Free memory on device (14.41/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 7.54 GiB for consumed memory (weights + non-torch), 0.87 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=4100118733` (3.82 GiB) to fit into requested memory, or `--kv-cache-memory=6285380608` (5.85 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.97 GiB.
(EngineCore pid=3478) WARNING 11-21 05:42:47 [extensible_kv_cache.py:662] g

##**Step 5**

Leverage the loaded model via VLLM to inference a sample image

In [3]:
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
            {"type": "text", "text": "Describe this image in two sentences."},
        ],
    }
]

prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, _ = process_vision_info(messages)

sampling_params = SamplingParams(temperature=0.2, max_tokens=256)

outputs = llm.generate(
    {
        "prompt": prompt,
        "multi_modal_data": {"image": image_inputs},
    },
    sampling_params=sampling_params,
)

print(outputs[0].outputs[0].text)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.16s/it, est. speed input: 169.30 toks/s, output: 8.45 toks/s]

A woman and her golden retriever are sitting on the beach, facing each other. The woman is wearing a plaid shirt and black pants, while the dog is wearing a colorful harness. They are both smiling and appear to be enjoying their time together.
